# M3L4 E04 — Golden Dataset para routing [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

Hasta ahora debuggeamos traces **reactivamente**: el problema ya ocurrió y lo analizamos. Pero en ingeniería de IA queremos **medir calidad antes de liberar a producción**.

Un **Golden Dataset** es un conjunto de casos etiquetados manualmente que sirven como "verdad ground truth". Ejecutamos nuestro sistema contra el dataset y medimos:
- **Accuracy**: qué porcentaje de casos clasifica correctamente
- **Fallos por dominio**: qué áreas del negocio tienen peor performance
- **Precisión por intent**: hay dominios más fáciles o más difíciles?

| Concepto | Definición simple | Cómo aparece acá |
|---|---|---|
| **Golden Dataset** | Casos de prueba con respuesta esperada | Lista de `{id, query, expected_intent}` |
| **Routing accuracy** | % de queries donde el intent detectado coincide con el esperado | `df['correct'].mean()` |
| **Fallos por dominio** | Casos incorrectos agrupados por intent | `df[df.correct==0].groupby('expected_intent')` |
| **Ground truth** | La respuesta correcta definida por un humano | `expected_intent` en cada caso |

In [ ]:
!pip install pandas -q
import pandas as pd

def route_query(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ['vacaciones', 'licencia', 'recibo', 'portal rrhh']):
        return 'hr'
    if any(w in q for w in ['vpn', 'laptop', 'error', 'app', 'wifi', 'login']):
        return 'it'
    if any(w in q for w in ['factura', 'pago', 'reembolso']):
        return 'finance'
    if any(w in q for w in ['contrato', 'legal', 'confidencialidad', 'nda']):
        return 'legal'
    if len(q.split()) <= 2:
        return 'clarification'
    return 'general'

golden_dataset = [
    {'id': 'case_001', 'query': '¿Cómo solicito mis días de vacaciones?',          'expected_intent': 'hr'},
    {'id': 'case_002', 'query': 'Mi VPN no conecta desde ayer',                     'expected_intent': 'it'},
    {'id': 'case_003', 'query': 'Necesito ver mi factura del mes pasado',            'expected_intent': 'finance'},
    {'id': 'case_004', 'query': 'Necesito el contrato de confidencialidad actualizado', 'expected_intent': 'legal'},
    {'id': 'case_005', 'query': 'No puedo entrar al portal para ver mi recibo',     'expected_intent': 'hr'},
    {'id': 'case_006', 'query': 'ayuda',                                             'expected_intent': 'clarification'},
    {'id': 'case_007', 'query': '¿Cuándo se procesa el reembolso de gastos?',       'expected_intent': 'finance'},
    {'id': 'case_008', 'query': 'El sistema de login no me deja entrar',             'expected_intent': 'it'},
]
print('Setup listo.')

In [ ]:
def evaluate_router(router_fn, dataset: list) -> tuple:
    rows = []
    for case in dataset:
        actual_intent = router_fn(case['query'])
        correct = int(actual_intent == case['expected_intent'])
        rows.append({
            'id': case['id'],
            'query': case['query'],
            'expected_intent': case['expected_intent'],
            'actual_intent': actual_intent,
            'correct': correct
        })
    df = pd.DataFrame(rows)
    accuracy = df['correct'].mean()
    return df, accuracy

print('Función lista.')

## Solución — Evaluación del router

Ejecutamos el router contra los 8 casos del golden dataset. La métrica principal es **routing accuracy**.

In [ ]:
df, accuracy = evaluate_router(route_query, golden_dataset)
print(f'Routing accuracy: {accuracy:.2%}')
df

## Análisis de fallos

Veamos qué casos fallaron. Esto nos dice **dónde mejorar** el router.

In [ ]:
df_failures = df[df['correct'] == 0]
print(f'Fallos: {len(df_failures)}')
df_failures

## Precisión por dominio

No todos los dominios son igual de fáciles. Algunos pueden tener palabras clave más ambiguas.

In [ ]:
accuracy_by_intent = df.groupby('expected_intent')['correct'].mean().sort_values()
print('Accuracy por intent:')
accuracy_by_intent

## Verificación

In [ ]:
assert df is not None
assert 0 <= accuracy <= 1
assert 'correct' in df.columns
assert 'actual_intent' in df.columns
print(f'Checks E04 OK — Routing accuracy: {accuracy:.2%}')

## [OK] Cierre — ¿Qué logramos?

| Métrica | Valor | Indica |
|---|---|---|
| **Routing accuracy** | Proporción de aciertos | Calidad global del router |
| **Fallos totales** | Casos incorrectos | Cuántos usuarios reciben respuestas equivocadas |
| **Accuracy por intent** | Precisión por dominio | Qué áreas necesitan mejorar |

> **Dato importante:** sin un golden dataset, no tenés forma objetiva de saber si tu sistema está mejorando o empeorando. Es la base de cualquier ciclo de mejora.

**¿Qué sigue?** En E05 vamos a comparar dos versiones del router usando este golden dataset para ver cuál es mejor.